# Compte de résultat prévisionnel trimestriel avec PROC COMPUTAB

## Résumé

Ce notebook construit le compte de résultat prévisionnel trimestriel d'une banque régionale directement à partir de données de grand livre mensuelles à l'aide de PROC COMPUTAB, la procédure de production de tableaux de rapport de SAS/ETS. Il oriente les produits d'intérêts, les charges d'intérêts, les produits de commissions et les coûts d'exploitation de chaque mois vers la bonne colonne de trimestre calendaire, puis utilise des blocs de programmation de lignes pour calculer le produit net d'intérêts, le résultat avant impôt et le résultat net, et un bloc de colonne pour agréger les quatre trimestres en un total annuel.

## Sources de données

L'unique jeu de données synthétique `bank_ledger` simule un exercice comptable de postes d'états financiers mensuels pour une banque communautaire de taille moyenne. Douze observations mensuelles sont générées en ligne avec `call streaminit`/`rand` afin que le notebook soit entièrement autonome.

| Variable | Type | Description |
|----------|------|-------------|
| `stmtdate` | Num (DATE9.) | Date de clôture mensuelle de l'état (janv.–déc.) |
| `loanint`  | Num | Intérêts et commissions perçus sur le portefeuille de prêts (milliers USD) |
| `secint`   | Num | Intérêts perçus sur le portefeuille de titres de placement (milliers USD) |
| `depint`   | Num | Intérêts versés sur les dépôts (milliers USD) |
| `borrint`  | Num | Intérêts versés sur les emprunts / avances FHLB (milliers USD) |
| `feeinc`   | Num | Produits hors intérêts (commissions et frais de service) (milliers USD) |
| `salaries` | Num | Charges de salaires et avantages du personnel (milliers USD) |
| `occupancy`| Num | Charges d'occupation et d'équipement (milliers USD) |
| `othropex` | Num | Autres charges d'exploitation hors intérêts (milliers USD) |
| `provision`| Num | Provision pour pertes sur crédit (milliers USD) |
| `taxrate`  | Num | Taux d'imposition effectif appliqué au résultat avant impôt |

# Compte de résultat prévisionnel trimestriel avec PROC COMPUTAB

Les équipes financières bancaires transforment couramment un grand livre mensuel en un **compte de résultat trimestriel** qui présente le produit net d'intérêts et le résultat net final. `PROC COMPUTAB` (SAS/ETS) est conçu à cet effet : il dispose un tableau programmable dont les **colonnes** sont les périodes de reporting et les **lignes** les postes d'état, et il vous permet de calculer des sous-totaux avec la logique ordinaire de l'étape DATA dans des blocs de lignes et de colonnes.

Dans ce notebook, nous :

1. Générons un exercice de données de grand livre mensuelles synthétiques pour une banque communautaire.
2. Orientons chaque mois vers son trimestre calendaire et construisons un compte de résultat trimestriel.
3. Calculons le produit net d'intérêts, le résultat avant impôt et le résultat net dans un **bloc de lignes**, et agrégeons les trimestres en un total annuel dans un **bloc de colonne**.
4. Réutilisons le tableau `out=` afin que l'état calculé puisse alimenter le reporting en aval.

## Étape 1 — Générer des données de grand livre mensuelles synthétiques

Nous simulons douze observations de clôture mensuelle. Les produits d'intérêts sur prêts et titres progressent doucement au fil de l'année, les coûts des dépôts et des emprunts évoluent avec l'environnement de taux, et les produits de commissions, les charges d'exploitation et la provision pour pertes sur crédit portent un bruit mensuel réaliste. `call streaminit` fixe la graine afin que les données soient reproductibles.

In [1]:
DONNÉES bank_ledger;
   APPELER streaminit(20240115);
   format stmtdate date9.;
   FAIRE month = 1 JUSQU_À 12;
      /* Month-end statement date for fiscal year 2024 */
      stmtdate = mdy(month, 28, 2024);

      /* Mild upward drift across the year + monthly noise */
      drift = 1 + 0.012 * (month - 1);

      /* Interest income (USD thousands) */
      loanint = round(1850 * drift + rand('normal', 0, 45), 0.01);
      secint  = round( 420 * drift + rand('normal', 0, 15), 0.01);

      /* Interest expense (USD thousands) */
      depint  = round( 540 * drift + rand('normal', 0, 22), 0.01);
      borrint = round( 130 * drift + rand('normal', 0, 10), 0.01);

      /* Non-interest income and expense (USD thousands) */
      feeinc    = round(310 + rand('normal', 0, 18), 0.01);
      salaries  = round(720 + rand('normal', 0, 25), 0.01);
      occupancy = round(165 + rand('normal', 0, 8),  0.01);
      othropex  = round(240 + rand('normal', 0, 20), 0.01);

      /* Provision for credit losses, occasionally elevated */
      provision = round(95 + rand('exponential') * 40, 0.01);

      /* Effective tax rate */
      taxrate = 0.21;

      SORTIE;
   FIN;
   GARDER stmtdate loanint secint depint borrint
        feeinc salaries occupancy othropex provision taxrate;
EXÉCUTER;

PROCÉDURE IMPRIMER DONNÉES=bank_ledger noobs ÉTIQUETTE;
   TITRE 'Synthetic Monthly Ledger — Community Bank FY2024';
   format loanint secint depint borrint feeinc
          salaries occupancy othropex provision comma8.2;
EXÉCUTER;

                                    Synthetic Monthly Ledger — Community Bank FY2024                                    

 STMTDATE   LOANINT  SECINT  DEPINT  BORRINT  FEEINC  SALARIES  OCCUPANCY  OTHROPEX  PROVISION  TAXRATE
28JAN2024  1,772.95  423.79  531.47   128.99  339.33    699.38     171.95    202.43     130.93     0.21
28FEB2024  1,824.38  421.13  564.85   125.53  294.09    767.29     162.79    307.61     123.25     0.21
28MAR2024  1,931.98  442.06  536.64   131.71  295.72    724.03     153.26    254.21     115.76     0.21
28APR2024  1,934.99  439.29  535.94   140.01  294.56    729.47     174.19    237.43     198.05     0.21
28MAY2024  1,949.31  447.35  591.44   124.42  299.78    739.13     165.08    223.32     105.57     0.21
28JUN2024  1,934.36  448.28  551.70   147.64  335.81    718.90     171.91    236.94     130.13     0.21
28JUL2024  1,936.57  439.22  565.70   133.82  293.21    743.87     174.16    247.19     199.20     0.21
28AUG2024  1,979.73  457.42  558.54   144.45  

## Étape 2 — Construire le compte de résultat trimestriel

Le cœur du rapport est l'étape `PROC COMPUTAB` ci-dessous.

- **`columns qtr1 qtr2 qtr3 qtr4 fy;`** définit quatre colonnes de trimestre plus une colonne annuelle.
- Le **bloc d'entrée** non libellé règle la variable automatique **`_col_`** avec `qtr(stmtdate)`, qui oriente chaque observation mensuelle vers la bonne colonne de trimestre. Comme l'entrée est transposée par défaut, chaque variable de grand livre se place sur la ligne qui partage son nom.
- Le **bloc de lignes** `inc_stmt:` s'exécute une fois par colonne et calcule les lignes dérivées — produit net d'intérêts, total des charges hors intérêts, résultat avant impôt et résultat net — exactement comme le ferait un comptable.
- Le **bloc de colonne** `total:` s'exécute une fois par ligne et somme les quatre trimestres dans la colonne `fy` (annuelle).

Les instructions `rows ... / ...` ajoutent des titres, une indentation et des filets (`ol` filet supérieur, `ul` filet inférieur, `dul` double filet inférieur) afin que la sortie se lise comme un véritable état financier.

In [2]:
TITRE 'Pro Forma Income Statement — Community Bancorp, Inc.';
title2 'Fiscal Year 2024  (Amounts in USD Thousands)';

PROCÉDURE computab DONNÉES=bank_ledger cspace=2 cwidth=11 out=qtr_income;

   /* Four quarters plus a rolled-up full-year column */
   columns qtr1 qtr2 qtr3 qtr4 fy / format=comma11.1;
   columns qtr1 / 'Q1';
   columns qtr2 / 'Q2';
   columns qtr3 / 'Q3';
   columns qtr4 / 'Q4';
   columns fy   / 'Full' 'Year' +3;

   /* Income statement rows, top to bottom */
   rows loanint  / 'Interest & Fees on Loans';
   rows secint   / 'Interest on Securities'        ul;
   rows intinc   / 'Total Interest Income';
   rows depint   / 'Interest on Deposits';
   rows borrint  / 'Interest on Borrowings'        ul;
   rows intexp   / 'Total Interest Expense';
   rows nii      / 'Net Interest Income'           ol skip;
   rows provision/ 'Provision for Credit Losses'   ul;
   rows niiap    / 'Net Int. Income after Prov.'   skip;
   rows feeinc   / 'Non-Interest Income'           ul;
   rows salaries / 'Salaries & Benefits';
   rows occupancy/ 'Occupancy & Equipment';
   rows othropex / 'Other Operating Expense'       ul;
   rows nonintexp/ 'Total Non-Interest Expense'    skip;
   rows pretax   / 'Pre-Tax Income'                ol;
   rows taxexp   / 'Income Tax Expense'            ul;
   rows netinc   / 'Net Income'                    dul;

   /* Input block: route each month to its calendar quarter */
   _col_ = qtr(stmtdate);

   /* Row block: compute statement subtotals across every column */
   inc_stmt:
      intinc    = loanint + secint;
      intexp    = depint + borrint;
      nii       = intinc - intexp;
      niiap     = nii - provision;
      nonintexp = salaries + occupancy + othropex;
      pretax    = niiap + feeinc - nonintexp;
      taxexp    = pretax * 0.21;
      netinc    = pretax - taxexp;

   /* Column block: roll the four quarters into the full-year column */
   TOTAL:
      fy = qtr1 + qtr2 + qtr3 + qtr4;
EXÉCUTER;

                                  Pro Forma Income Statement — Community Bancorp, Inc.                                  
                                      Fiscal Year 2024  (Amounts in USD Thousands)                                      


                             The COMPUTAB Procedure                             

                             Q1           Q2           Q3           Q4         Full  
                                                                               Year  
                           qtr1         qtr2         qtr3         qtr4           fy  
                    -----------  -----------  -----------  -----------  -----------  
  Interest & Fees on Loans
  loanint               5529.31      5818.66      5963.46      6276.96     23588.39  
  Interest on Securities
  secint                1286.98      1334.92      1342.03      1452.88      5416.81  
                    -----------  -----------  -----------  -----------  -----------  
  Total Interest Inc

## Étape 3 — Réutiliser le jeu de données de sortie de COMPUTAB

L'étape `PROC COMPUTAB` ci-dessus a écrit son tableau calculé dans `qtr_income` via `out=`. Chaque ligne de ce jeu de données est un poste d'état et chaque variable de colonne (`qtr1`–`qtr4`, `fy`) contient la valeur calculée, de sorte qu'il peut alimenter le reporting en aval. Ci-dessous, nous imprimons la colonne annuelle agrégée pour confirmer que les chiffres se recoupent.

In [3]:
TITRE 'COMPUTAB Output Dataset — Full-Year Column';

PROCÉDURE IMPRIMER DONNÉES=qtr_income ÉTIQUETTE noobs;
   VAR _row_ fy;
   format fy comma12.1;
   ÉTIQUETTE _row_ = 'Line Item' fy = 'Full Year';
EXÉCUTER;

TITRE;

                                       COMPUTAB Output Dataset — Full-Year Column                                       
                                      Fiscal Year 2024  (Amounts in USD Thousands)                                      


Line Item  Full Year
---------  ---------
loanint     23,588.4
secint       5,416.8
intinc      29,005.2
depint       6,831.2
borrint      1,650.7
intexp       8,482.0
nii         20,523.2
provision    1,568.9
niiap       18,954.3
feeinc       3,703.2
salaries     8,763.1
occupancy    1,985.6
othropex     2,944.8
nonintexp   13,693.5
pretax       8,964.1
taxexp       1,882.5
netinc       7,081.6

NOTE: Option TITLE changed to COMPUTAB Output Dataset — Full-Year Column.
NOTE: PROC PRINT data=qtr_income

NOTE: PROC PRINT completed: 17 observations printed, 2 variables


## Interprétation des résultats

L'état prévisionnel se lit de haut en bas comme le call report réglementaire d'une banque : le total des produits d'intérêts moins le total des charges d'intérêts donne le **produit net d'intérêts** — ici environ \$20,5 millions pour l'année — le principal moteur de résultat de la banque. En soustrayant la **provision pour pertes sur crédit**, en ajoutant les **produits de commissions** et en déduisant les **charges hors intérêts**, on obtient un résultat avant impôt d'environ \$9,0 millions, et l'application du taux d'imposition effectif de 21 % produit un **résultat net** proche de \$7,1 millions. Le bloc de colonne `total:` additionne les quatre trimestres dans la colonne annuelle, de sorte que les totaux annuels se rapprochent de la somme des trimestres par construction — confirmé dans le jeu de données `out=`, où le `netinc` annuel de 7 081,6 égale la somme des quatre chiffres trimestriels.

Comme tout est calculé à l'intérieur des blocs de lignes et de colonnes programmables de `PROC COMPUTAB`, substituer un véritable grand livre mensuel ne nécessite aucun changement de la logique du rapport — seul le jeu de données d'entrée change. Le jeu de données `out=` rend ensuite l'état calculé disponible pour des tableaux de bord, l'analyse de tendance ou l'automatisation du dossier de conseil d'administration.